### Notebook 范围

### 目的
计算 FWD 和 INT/NOCOUPL skill，并生成最终 synthesis matrix。

### 科学问题
interactive O3 是否延迟 50 hPa final warming？是否改变 O3 pathway skill？哪些机制证据在多 case 中稳健？

### 方法
FWD 明确定义为 U60N50 < 7 m/s 并持续 10 天的第一天。O3 skill 同时给出 init-May30 RMSE、Mar-Apr RMSE 和 Mar-Apr minimum-O3 RMSE。最终矩阵只显示数值和 NA，不再使用 √ 或 ?。

### 输出
outputs/figures/08_final_warming_skill 和 outputs/figures/99_synthesis。

### 解读
FWD 正差值表示 INT 比 NOCOUPL 更晚 final warming；RMSE 负差值表示 INT 误差更小。

### 注意
FWD 若在 hindcast 时段内未跨阈值则为 NA；skill 是相对于 BWCN same-year reference，不等同 Marina 复现。


### 导入与公共路径

### 目的
加载 Extention_analysis 公共函数。

### 科学问题
所有 notebook 是否共享相同路径、变量定义和 sign convention？

### 方法
导入 hindcast_extension_utils.py。

### 输出
无图。

### 解读
路径正确即可继续。

### 注意
所有输出都写入 outputs 子目录。


In [ ]:
from pathlib import Path
import sys

WORK_ROOT = Path("/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/Extention_analysis")
if str(WORK_ROOT) not in sys.path:
    sys.path.insert(0, str(WORK_ROOT))

import math
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import hindcast_extension_utils as hx
from hindcast_extension_utils import *

for d in [FIG_DIR, TAB_DIR, CACHE_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT", WORK_ROOT)
print("HINDCAST_ROOT", HINDCAST_ROOT)


### 计算 FWD 和 skill

### 目的
对 paired INT/NOCOUPL small-perturbation cases 计算 FWD distribution、O3 skill 均值差和 ensemble spread 差异。

### 科学问题
interactive O3 是否让 50 hPa 或 1 hPa final warming 更晚？它改变的是相对于 BWCN reference 的平均误差，还是 ensemble spread？

### 方法
FWD 直接读取 `final_warming_date/<case>_FWD_plev_member.nc` 中的 `FWD_dayofyear`，分别取 50 hPa 和 1 hPa。该产品的定义是：第一个低于压力相关阈值的 Jan-Jun forecast day，之后不再出现连续 10 天恢复西风；阈值为 `0 m/s` for `p <= 10 hPa`、`7 m/s` for `p > 10 hPa`。左图是 50 hPa FWD distribution，右图是 1 hPa FWD distribution；两者都使用 INT 和 NOCOUPL 的 ensemble boxplot，不做 member-by-member subtraction。O3 skill 使用 O3_RMSE_init_May30、O3_RMSE_MarApr、O3_min_RMSE_MarApr，均相对同年 BWCN reference；其中 O3_min_RMSE_MarApr = sqrt((member MA-min O3 - reference MA-min O3)^2)，避免 signed minimum error 在 INT-NOCOUPL 中退化成 MA-min O3 直接相减。INT-NOCOUPL 表示 ensemble mean(INT) - ensemble mean(NOCOUPL)，spread 图表示 ensemble std(INT) - ensemble std(NOCOUPL)。

### 输出
08_FWD_distributions_INT_vs_NOCOUPL_v3、08_INT_vs_NOCOUPL_skill_mean_difference_heatmap_v4、08_INT_vs_NOCOUPL_skill_spread_difference_heatmap_v4，以及对应 CSV 表。

### 解读
FWD 图中 y 轴显示直观月日刻度；黑色实线是 ensemble mean，灰色虚线是 median；线越高表示 final warming 越晚。skill mean difference 中负 RMSE 差值表示 INT 平均误差更小；spread difference 中正值表示 INT ensemble 离散度更大。

### 注意
INT 和 NOCOUPL 成员并非严格一一对应，因此这里不做 paired member subtraction。1 hPa panel 是垂直结构对照，不替代标准 50 hPa FWD 定义。

In [ ]:
fig_dir = figure_dir("08_final_warming_skill")
tab_dir = table_dir("08_final_warming_skill")
inv = discover_hindcast_cases()
pairs = paired_int_nocoupl_cases(inv)

FWD_LEVELS = [50, 1]
FWD_LEVEL_LABELS = {50: "50 hPa", 1: "1 hPa"}
O3_SKILL_METRICS = ["O3_RMSE_init_May30", "O3_RMSE_MarApr", "O3_min_RMSE_MarApr"]
O3_SKILL_LABELS = {
    "O3_RMSE_init_May30": "O3 RMSE\ninit-May30",
    "O3_RMSE_MarApr": "O3 RMSE\nMar-Apr",
    "O3_min_RMSE_MarApr": "MA min O3\nRMSE",
}
CONFIG_LABELS = {"INT": "H-INT-3D", "NOCOUPL": "H Clim 3D"}
CONFIG_COLORS = {"INT": "#1f77b4", "NOCOUPL": "#ff7f0e"}


def pair_label(year, init):
    return f"{int(year):02d}-{int(init):02d}"


def paired_case_label(row):
    return f"{pair_label(row['year'], row['init_month'])}"


def finite_mean_std(values):
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.nan, np.nan, 0
    std = float(np.std(arr, ddof=1)) if arr.size > 1 else 0.0
    return float(np.mean(arr)), std, int(arr.size)


FWD_MONTH_TICKS = [32, 60, 91, 121, 152, 182]
FWD_MONTH_LABELS = ["Feb 1", "Mar 1", "Apr 1", "May 1", "Jun 1", "Jul 1"]


def set_fwd_month_yaxis(ax, values):
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size:
        lo = max(45.0, np.floor((finite.min() - 8.0) / 5.0) * 5.0)
        hi = min(190.0, np.ceil((finite.max() + 8.0) / 5.0) * 5.0)
    else:
        lo, hi = 55.0, 180.0
    ticks = [t for t in FWD_MONTH_TICKS if lo <= t <= hi]
    labels = [FWD_MONTH_LABELS[FWD_MONTH_TICKS.index(t)] for t in ticks]
    ax.set_ylim(lo, hi)
    ax.set_yticks(ticks)
    ax.set_yticklabels(labels)
    ax.set_ylabel("Final warming date")


fwd_rows = []
skill_rows = []
for _, pair in pairs.iterrows():
    for config, case in [("INT", pair["INT_case"]), ("NOCOUPL", pair["NOCOUPL_case"])]:
        year = parse_case_name(case)["year"]
        fwd_product = load_fwd_product(case)
        if fwd_product.empty:
            print(f"[WARN] {case}: missing FWD product; falling back to on-the-fly U60 calculation")
        for plev in FWD_LEVELS:
            if not fwd_product.empty:
                fwd = fwd_product.loc[np.isclose(fwd_product["plev_hpa"].astype(float), float(plev)), ["member", "FWD_DOY"]].copy()
            else:
                u, _ = compute_u60(case, plev)
                fwd = compute_fwd_from_u60n50(u)
            fwd["plev_hpa"] = float(plev)
            fwd["case"] = case
            fwd["config"] = config
            fwd["config_label"] = CONFIG_LABELS[config]
            fwd["year"] = pair["year"]
            fwd["init_month"] = pair["init_month"]
            fwd["pair_label"] = paired_case_label(pair)
            fwd_rows.append(fwd)

        o3, o3_date = load_hindcast_o3(case)
        ref, ref_date = load_bwcn_reference_o3(year)
        rmse_full = compute_o3_rmse(o3, ref, *target_window_for_case(case)).rename(columns={"O3_RMSE": "O3_RMSE_init_May30"})
        rmse_ma = compute_o3_rmse(o3, ref, (3, 1), (4, 30)).rename(columns={"O3_RMSE": "O3_RMSE_MarApr"})
        ma_min = o3_ma_min(o3, o3_date).rename(columns={"O3_MA_min": "O3_MA_min_member"})
        ref_min = o3_ma_min(ref, ref_date).rename(columns={"O3_MA_min": "O3_MA_min_reference"})
        ref_val = float(ref_min["O3_MA_min_reference"].iloc[0]) if not ref_min.empty else np.nan
        skill = rmse_full.merge(rmse_ma, on="member", how="outer").merge(ma_min, on="member", how="outer")
        skill["O3_min_error_MarApr"] = skill["O3_MA_min_member"] - ref_val
        skill["O3_min_RMSE_MarApr"] = np.sqrt(skill["O3_min_error_MarApr"] ** 2)
        skill["case"] = case
        skill["config"] = config
        skill["config_label"] = CONFIG_LABELS[config]
        skill["year"] = pair["year"]
        skill["init_month"] = pair["init_month"]
        skill["pair_label"] = paired_case_label(pair)
        skill_rows.append(skill)

fwd_df = pd.concat(fwd_rows, ignore_index=True) if fwd_rows else pd.DataFrame()
skill_df = pd.concat(skill_rows, ignore_index=True) if skill_rows else pd.DataFrame()
fwd_csv = tab_dir / "08_FWD_INT_NOCOUPL_member_summary.csv"
skill_csv = tab_dir / "08_INT_vs_NOCOUPL_skill_member_metrics.csv"
fwd_df.to_csv(fwd_csv, index=False)
skill_df.to_csv(skill_csv, index=False)
# Backward-compatible copies used by the synthesis block.
fwd_df.to_csv(tab_dir / "08_FWD_INT_NOCOUPL_summary.csv", index=False)
skill_df.to_csv(tab_dir / "08_INT_vs_NOCOUPL_skill_metrics.csv", index=False)
fwd_df.to_csv(TAB_DIR / "08_FWD_INT_NOCOUPL_summary.csv", index=False)
skill_df.to_csv(TAB_DIR / "08_INT_vs_NOCOUPL_skill_metrics.csv", index=False)

# --- FWD distributions: non-paired ensemble boxplots at 50 and 1 hPa.
fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8), sharey=False, constrained_layout=False)
fig.subplots_adjust(left=0.08, right=0.98, top=0.86, bottom=0.23, wspace=0.22)
if fwd_df.empty:
    for ax in axes:
        ax.axis("off")
        ax.text(0.5, 0.5, "No paired INT/NOCOUPL FWD data", ha="center", va="center")
else:
    labels = [pair_label(y, m) for y, m in sorted(fwd_df[["year", "init_month"]].drop_duplicates().itertuples(index=False, name=None), key=lambda x: (int(x[0]), int(x[1])))]
    positions = np.arange(len(labels), dtype=float)
    width = 0.28
    for ax, plev in zip(axes, FWD_LEVELS):
        sub_level = fwd_df.loc[fwd_df["plev_hpa"] == float(plev)].copy()
        for cfg, offset in [("INT", -width / 1.8), ("NOCOUPL", width / 1.8)]:
            grouped = []
            pos = []
            for i, lab in enumerate(labels):
                vals = sub_level.loc[(sub_level["pair_label"] == lab) & (sub_level["config"] == cfg), "FWD_DOY"].dropna().values
                grouped.append(vals if len(vals) else np.array([np.nan]))
                pos.append(positions[i] + offset)
            bp = ax.boxplot(
                grouped,
                positions=pos,
                widths=width,
                patch_artist=True,
                showfliers=False,
                showmeans=True,
                meanline=True,
                meanprops={"color": "black", "linewidth": 1.8, "linestyle": "-"},
                medianprops={"color": "0.25", "linewidth": 1.2, "linestyle": ":"},
                boxprops={"linewidth": 1.0},
                whiskerprops={"linewidth": 0.9},
                capprops={"linewidth": 0.9},
            )
            for box in bp["boxes"]:
                box.set_facecolor(CONFIG_COLORS[cfg])
                box.set_alpha(0.72)
            ax.plot([], [], color=CONFIG_COLORS[cfg], lw=8, alpha=0.72, label=CONFIG_LABELS[cfg])
        ax.set_xticks(positions, labels, rotation=35, ha="right")
        set_fwd_month_yaxis(ax, sub_level["FWD_DOY"].values)
        ax.set_title(f"{FWD_LEVEL_LABELS[plev]} final warming distribution")
        ax.grid(True, axis="y", alpha=0.35)
        ax.grid(False, axis="x")
        ax.text(0.01, 0.98, "Box = ensemble distribution; black solid = mean; gray dotted = median", transform=ax.transAxes, ha="left", va="top", fontsize=9)
    handles, legend_labels = axes[0].get_legend_handles_labels()
    fig.legend(handles[:2], legend_labels[:2], loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.03))
    fig.suptitle("INT vs NOCOUPL final warming dates (no member-by-member subtraction)", fontsize=13, fontweight="bold")
savefig(
    fig,
    "08_FWD_distributions_INT_vs_NOCOUPL_v3",
    fig_dir=fig_dir,
    notebook="08_final_warming_and_skill.ipynb",
    scientific_question="interactive O3 是否延迟 50 hPa 或 1 hPa final warming？",
    variables_windows="FWD read from final_warming_date product: first below threshold with no later 10-day westerly return; thresholds 0 m/s for p<=10 hPa and 7 m/s for p>10 hPa",
    interpretation="y 轴使用月日标签；箱体、黑色均值线和灰色中位数线越高表示 final warming 越晚；左右图分别是 50 hPa 和 1 hPa。",
    caveat="INT/NOCOUPL members are not assumed one-to-one; 1 hPa panel is a vertical-context diagnostic and uses the official product threshold convention.",
    csv_table=fwd_csv,
)
plt.show()

# --- Non-paired ensemble mean and spread differences for O3 skill.
skill_summary_rows = []
skill_delta_rows = []
skill_spread_rows = []
for (year, init), sub in skill_df.groupby(["year", "init_month"]) if not skill_df.empty else []:
    row_base = {"year": str(year).zfill(4), "init_month": str(init).zfill(2), "pair_label": pair_label(year, init)}
    for metric in O3_SKILL_METRICS:
        stats = {}
        for cfg in ["INT", "NOCOUPL"]:
            mean, std, n = finite_mean_std(sub.loc[sub["config"] == cfg, metric])
            stats[cfg] = {"mean": mean, "std": std, "n": n}
            skill_summary_rows.append({**row_base, "config": cfg, "config_label": CONFIG_LABELS[cfg], "metric": metric, "metric_label": O3_SKILL_LABELS[metric], "mean": mean, "std": std, "n": n})
        if stats["INT"]["n"] and stats["NOCOUPL"]["n"]:
            skill_delta_rows.append({
                **row_base,
                "metric": metric,
                "metric_label": O3_SKILL_LABELS[metric],
                "mean_INT": stats["INT"]["mean"],
                "mean_NOCOUPL": stats["NOCOUPL"]["mean"],
                "mean_INT_minus_NOCOUPL": stats["INT"]["mean"] - stats["NOCOUPL"]["mean"],
                "n_INT": stats["INT"]["n"],
                "n_NOCOUPL": stats["NOCOUPL"]["n"],
            })
            skill_spread_rows.append({
                **row_base,
                "metric": metric,
                "metric_label": O3_SKILL_LABELS[metric],
                "spread_INT": stats["INT"]["std"],
                "spread_NOCOUPL": stats["NOCOUPL"]["std"],
                "spread_INT_minus_NOCOUPL": stats["INT"]["std"] - stats["NOCOUPL"]["std"],
                "n_INT": stats["INT"]["n"],
                "n_NOCOUPL": stats["NOCOUPL"]["n"],
            })

# Add FWD mean and spread diagnostics so the mean and spread heatmaps use the same metric columns.
for (year, init, plev), sub in fwd_df.groupby(["year", "init_month", "plev_hpa"]) if not fwd_df.empty else []:
    metric = f"FWD_{int(plev)}hPa"
    metric_label = f"FWD\n{int(plev)} hPa"
    stats = {}
    for cfg in ["INT", "NOCOUPL"]:
        mean, std, n = finite_mean_std(sub.loc[sub["config"] == cfg, "FWD_DOY"])
        stats[cfg] = {"mean": mean, "std": std, "n": n}
    if stats["INT"]["n"] and stats["NOCOUPL"]["n"]:
        common = {
            "year": str(year).zfill(4),
            "init_month": str(init).zfill(2),
            "pair_label": pair_label(year, init),
            "metric": metric,
            "metric_label": metric_label,
            "n_INT": stats["INT"]["n"],
            "n_NOCOUPL": stats["NOCOUPL"]["n"],
        }
        skill_delta_rows.append({
            **common,
            "mean_INT": stats["INT"]["mean"],
            "mean_NOCOUPL": stats["NOCOUPL"]["mean"],
            "mean_INT_minus_NOCOUPL": stats["INT"]["mean"] - stats["NOCOUPL"]["mean"],
        })
        skill_spread_rows.append({
            **common,
            "spread_INT": stats["INT"]["std"],
            "spread_NOCOUPL": stats["NOCOUPL"]["std"],
            "spread_INT_minus_NOCOUPL": stats["INT"]["std"] - stats["NOCOUPL"]["std"],
        })

skill_summary = pd.DataFrame(skill_summary_rows)
skill_delta = pd.DataFrame(skill_delta_rows)
skill_spread = pd.DataFrame(skill_spread_rows)
skill_summary_csv = tab_dir / "08_INT_vs_NOCOUPL_skill_ensemble_summary.csv"
skill_delta_csv = tab_dir / "08_INT_vs_NOCOUPL_skill_mean_difference_heatmap.csv"
skill_spread_csv = tab_dir / "08_INT_vs_NOCOUPL_skill_spread_difference_heatmap.csv"
skill_summary.to_csv(skill_summary_csv, index=False)
skill_delta.to_csv(skill_delta_csv, index=False)
skill_spread.to_csv(skill_spread_csv, index=False)
# Backward-compatible copy for synthesis; now stores non-paired ensemble mean differences.
skill_delta.to_csv(tab_dir / "08_INT_vs_NOCOUPL_skill_difference_heatmap.csv", index=False)
skill_delta.to_csv(TAB_DIR / "08_INT_vs_NOCOUPL_skill_difference_heatmap.csv", index=False)


def plot_difference_heatmap(df, value_col, title, cbar_label, csv_path, fig_name, metric_order=None, center_zero=True):
    n_rows = int(df["pair_label"].nunique()) if (not df.empty and "pair_label" in df.columns) else 1
    fig, ax = plt.subplots(figsize=(11.8, max(4.2, 0.52 * max(1, n_rows) + 2.6)), constrained_layout=False)
    fig.subplots_adjust(left=0.15, right=0.88, top=0.82, bottom=0.34)
    if df.empty:
        ax.axis("off")
        ax.text(0.5, 0.5, "No paired INT/NOCOUPL statistics", ha="center", va="center")
    else:
        data = df.copy()
        if metric_order is None:
            metric_order = list(data["metric"].drop_duplicates())
        label_map = data.drop_duplicates("metric").set_index("metric")["metric_label"].to_dict()
        row_order = sorted(data["pair_label"].drop_duplicates(), key=lambda s: tuple(int(x) for x in s.split("-")))
        mat = data.pivot(index="pair_label", columns="metric", values=value_col).reindex(index=row_order, columns=metric_order)
        vals = mat.values.astype(float)
        finite = np.isfinite(vals)
        if finite.any():
            vmax = np.nanpercentile(np.abs(vals[finite]), 95)
            vmax = float(vmax) if np.isfinite(vmax) and vmax > 0 else float(np.nanmax(np.abs(vals[finite])))
        else:
            vmax = 1.0
        vmax = vmax if np.isfinite(vmax) and vmax > 0 else 1.0
        vmin = -vmax if center_zero else np.nanmin(vals[finite]) if finite.any() else -1.0
        im = ax.imshow(vals, cmap="RdBu_r", vmin=vmin, vmax=vmax, aspect="auto")
        ax.set_xticks(range(len(metric_order)), [label_map.get(m, m) for m in metric_order], rotation=0, ha="center")
        ax.tick_params(axis="x", labelsize=9, pad=8)
        ax.tick_params(axis="y", labelsize=9)
        ax.set_yticks(range(len(row_order)), row_order)
        ax.set_xlabel("Metric", labelpad=12)
        ax.set_ylabel("Hindcast year-init pair")
        ax.set_title(title, fontsize=12, fontweight="bold", pad=14)
        ax.set_xticks(np.arange(-0.5, len(metric_order), 1), minor=True)
        ax.set_yticks(np.arange(-0.5, len(row_order), 1), minor=True)
        ax.grid(which="minor", color="white", linewidth=1.6)
        ax.tick_params(which="minor", bottom=False, left=False)
        for i in range(len(row_order)):
            for j, metric in enumerate(metric_order):
                val = mat.iloc[i, j]
                if np.isfinite(val):
                    color = "white" if abs(val) > 0.62 * vmax else "black"
                    ax.text(j, i, f"{val:+.2f}", ha="center", va="center", fontsize=8.5, color=color)
                else:
                    ax.text(j, i, "NA", ha="center", va="center", fontsize=8.5, color="0.45")
        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.035)
        cb.set_label(cbar_label)
        fig.text(0.5, 0.07, "Values are ensemble statistics: H-INT-3D minus H Clim 3D. No member pairing is assumed.", ha="center", va="center", fontsize=9)
    savefig(
        fig,
        fig_name,
        fig_dir=fig_dir,
        notebook="08_final_warming_and_skill.ipynb",
        scientific_question="interactive O3 是否改变 O3 pathway skill、final warming date 或 ensemble spread？",
        variables_windows="O3 skill metrics relative to BWCN same-year reference; FWD from official FWD product; ensemble mean/std INT minus NOCOUPL",
        interpretation="负 RMSE mean difference 表示 INT 平均误差更小；MA-min O3 使用 scalar RMSE/absolute error 而不是 signed bias；正 FWD mean difference 表示 INT final warming 更晚；正 spread difference 表示 INT ensemble spread 更大。",
        caveat="统计量为两个 ensemble 的差，不是 member-paired subtraction；样本数随缺失 FWD 变化。",
        csv_table=csv_path,
    )
    plt.show()


mean_metric_order = O3_SKILL_METRICS + ["FWD_50hPa", "FWD_1hPa"]
plot_difference_heatmap(
    skill_delta,
    "mean_INT_minus_NOCOUPL",
    "INT - NOCOUPL ensemble mean differences",
    "ensemble mean difference (DU or days)",
    skill_delta_csv,
    "08_INT_vs_NOCOUPL_skill_mean_difference_heatmap_v4",
    metric_order=mean_metric_order,
)

spread_metric_order = mean_metric_order
plot_difference_heatmap(
    skill_spread,
    "spread_INT_minus_NOCOUPL",
    "INT - NOCOUPL ensemble spread differences",
    "ensemble std difference (DU or days)",
    skill_spread_csv,
    "08_INT_vs_NOCOUPL_skill_spread_difference_heatmap_v4",
    metric_order=spread_metric_order,
)


### INT 与 NOCOUPL spread onset 时间对比

### Purpose

这一个代码块在每一组同年同初始化的 H-INT-3D / H Clim 3D 中，分别计算两个 ensemble 的 spread onset，并同时输出相对初始化时间和绝对日历时间两张图。

### Scientific question

interactive O3 是否改变 dynamical spread 与 O3 spread 出现的时间？如果 H-INT-3D 的 vortex/O3 spread onset 系统性早于或晚于 H Clim 3D，说明 feedback 不只改变均值路径，也改变 ensemble 分歧开始的时间。

### Method

变量包括 O3 partial column、EP100 W1+W2、U60N10 和 U60N50。EP100 定义为 100 hPa、40-80N 平均的 `mean(-ep2)`，wave1+wave2 为两个分波的直接和；正值表示更强 upward wave activity。spread 定义为 member 标准差；onset 使用 `compute_spread_onset`：减去初始 spread 后标准化，5-day running mean 首次超过自身最大值的 50% 且维持 5 天。

### Expected output

输出 `08_INT_vs_NOCOUPL_spread_onset_lead_day_v1` 和 `08_INT_vs_NOCOUPL_spread_onset_calendar_month_day_v1` 的 png/pdf，以及 `08_INT_vs_NOCOUPL_spread_onset_summary.csv`。

### Interpretation

lead-day 图回答“初始化后几天开始分歧”；month-day 图回答“真实季节中哪一天开始分歧”。如果 EP/U onset 早于 O3 onset，则支持动力学 spread 是臭氧 spread 的上游来源。

### Caveat

INT 和 NOCOUPL 不做 member-by-member 配对，只比较各自 ensemble 的 spread onset。不同初始化月份的数据长度不同，缺失变量会跳过并写入日志。


In [ ]:
spread_fig_dir = figure_dir("08_final_warming_skill", "paired_spread_onset")
spread_tab_dir = table_dir("08_final_warming_skill")
SPREAD_LABELS = {
    "EP100_W1W2": "EP100 W1+W2",
    "U60N10": "U60N10",
    "U60N50": "U60N50",
    "O3": "O3 partial",
}
SPREAD_MARKERS = {"EP100_W1W2": "^", "U60N10": "o", "U60N50": "s", "O3": "D"}
SPREAD_COLORS = {"INT": CONFIG_COLORS["INT"], "NOCOUPL": CONFIG_COLORS["NOCOUPL"]}


def _ep100_w1w2_member_time(case):
    ep1, date1 = load_epflux(case, "wave1")
    ep2, date2 = load_epflux(case, "wave2")
    if ep1 is None or ep2 is None:
        log_message(f"{case}: missing wave1 or wave2 for 08 spread onset")
        return None, None
    n = min(ep1.sizes.get("lead_time", 0), ep2.sizes.get("lead_time", 0))
    if n == 0:
        return None, None
    da = -(ep1.isel(lead_time=slice(0, n)).sel(plev=10000.0, method="nearest") + ep2.isel(lead_time=slice(0, n)).sel(plev=10000.0, method="nearest"))
    da = coslat_weighted_mean(da, LAT_EP)
    date = np.asarray(date1)[:n]
    da = da.assign_coords(date=("lead_time", date))
    da.name = "EP100_W1W2"
    return da, date


def _spread_sources_for_case(case):
    sources = {}
    o3, o3_date = load_hindcast_o3(case)
    if o3 is not None:
        sources["O3"] = (o3, o3_date)
    ep, ep_date = _ep100_w1w2_member_time(case)
    if ep is not None:
        sources["EP100_W1W2"] = (ep, ep_date)
    for plev, name in [(10, "U60N10"), (50, "U60N50")]:
        u, u_date = compute_u60(case, plev)
        if u is not None:
            sources[name] = (u, u_date)
    return sources


spread_rows = []
for _, pair in pairs.iterrows():
    for config, case in [("INT", pair["INT_case"]), ("NOCOUPL", pair["NOCOUPL_case"])]:
        for variable, (da, date) in _spread_sources_for_case(case).items():
            onset = compute_spread_onset(da)
            lead = float(onset.get("onset_lead_day", np.nan))
            doy = case_time_doy(case, np.asarray(date)[: da.sizes["lead_time"]]) if date is not None else np.arange(da.sizes["lead_time"]) + init_doy_for_case(case)
            calendar_doy = float(doy[int(lead)]) if np.isfinite(lead) and int(lead) < len(doy) else np.nan
            spread_rows.append({
                "pair_label": paired_case_label(pair),
                "year": str(pair["year"]).zfill(4),
                "init_month": str(pair["init_month"]).zfill(2),
                "case": case,
                "config": config,
                "config_label": CONFIG_LABELS[config],
                "variable": variable,
                "variable_label": SPREAD_LABELS[variable],
                "onset_lead_day": lead,
                "onset_calendar_doy": calendar_doy,
                "threshold": float(onset.get("threshold", np.nan)),
                "n_member": int(da.sizes.get("member", 0)),
                "n_time": int(da.sizes.get("lead_time", da.sizes.get("time", 0))),
            })

spread_onset_df = pd.DataFrame(spread_rows)
spread_onset_csv = spread_tab_dir / "08_INT_vs_NOCOUPL_spread_onset_summary.csv"
spread_onset_df.to_csv(spread_onset_csv, index=False)
spread_onset_df.to_csv(TAB_DIR / "08_INT_vs_NOCOUPL_spread_onset_summary.csv", index=False)


def _plot_spread_onset(df, x_col, name, title, xlabel, calendar_axis=False):
    n_rows = df[["pair_label", "config"]].drop_duplicates().shape[0] if not df.empty else 1
    fig, ax = plt.subplots(figsize=(12.0, max(4.8, 0.42 * max(1, n_rows) + 2.0)), constrained_layout=False)
    fig.subplots_adjust(left=0.22, right=0.98, top=0.86, bottom=0.24)
    if df.empty:
        ax.axis("off")
        ax.text(0.5, 0.5, "No spread-onset data", ha="center", va="center")
    else:
        ordered_pairs = sorted(df["pair_label"].drop_duplicates(), key=lambda s: tuple(int(x) for x in s.split("-")))
        row_keys = []
        row_labels = []
        for pair_label_value in ordered_pairs:
            for cfg in ["INT", "NOCOUPL"]:
                sub = df[(df["pair_label"] == pair_label_value) & (df["config"] == cfg)]
                if sub.empty:
                    continue
                row_keys.append((pair_label_value, cfg))
                row_labels.append(f"{pair_label_value} {CONFIG_LABELS[cfg]}")
        y_pos = {key: i for i, key in enumerate(row_keys)}
        for _, row in df.iterrows():
            key = (row["pair_label"], row["config"])
            if key not in y_pos or not np.isfinite(row[x_col]):
                continue
            ax.scatter(
                row[x_col],
                y_pos[key],
                marker=SPREAD_MARKERS.get(row["variable"], "o"),
                s=82,
                color=SPREAD_COLORS.get(row["config"], "0.4"),
                edgecolor="black",
                linewidth=0.5,
                alpha=0.92,
                zorder=3,
            )
        ax.set_yticks(range(len(row_labels)), row_labels)
        ax.invert_yaxis()
        ax.set_xlabel(xlabel)
        ax.set_title(title, fontsize=12.5, fontweight="bold", pad=12)
        ax.grid(True, axis="x", alpha=0.35)
        ax.grid(False, axis="y")
        if calendar_axis:
            ticks = [1, 32, 60, 91, 121, 152]
            labels = ["Jan 1", "Feb 1", "Mar 1", "Apr 1", "May 1", "Jun 1"]
            ax.set_xlim(1, 170)
            ax.set_xticks(ticks, labels)
        else:
            xvals = df[x_col].values.astype(float)
            xmax = np.nanmax(xvals) if np.isfinite(xvals).any() else 120
            ax.set_xlim(-2, max(45, float(xmax) + 6))
        variable_handles = [Line2D([0], [0], marker=SPREAD_MARKERS[v], color="none", markerfacecolor="0.65", markeredgecolor="black", markersize=8, label=SPREAD_LABELS[v]) for v in ["EP100_W1W2", "U60N10", "U60N50", "O3"]]
        config_handles = [Line2D([0], [0], marker="o", color="none", markerfacecolor=SPREAD_COLORS[cfg], markeredgecolor="black", markersize=8, label=CONFIG_LABELS[cfg]) for cfg in ["INT", "NOCOUPL"]]
        ax.legend(handles=variable_handles + config_handles, loc="lower center", bbox_to_anchor=(0.5, -0.30), ncol=3, frameon=False, fontsize=9)
        ax.text(0.01, 0.01, "Onset = first 5-day running spread anomaly above 50% of its maximum for at least 5 days", transform=ax.transAxes, ha="left", va="bottom", fontsize=8.8)
    savefig(
        fig,
        name,
        fig_dir=spread_fig_dir,
        notebook="08_final_warming_and_skill.ipynb",
        scientific_question="interactive O3 是否改变 dynamical/O3 ensemble spread onset timing？",
        variables_windows="O3 partial 60-90N 30-70hPa; EP100 W1+W2 mean(-ep2) at 100hPa 40-80N; U60N10/U60N50; onset from member spread.",
        interpretation="lead-day 图看初始化后分歧时间；calendar 图看季节中实际分歧时间。EP/U onset 早于 O3 onset 支持 dynamics-first pathway。",
        caveat="INT/NOCOUPL 不做 member 配对；缺失变量跳过。",
        csv_table=spread_onset_csv,
    )
    plt.show()


_plot_spread_onset(
    spread_onset_df,
    "onset_lead_day",
    "08_INT_vs_NOCOUPL_spread_onset_lead_day_v1",
    "INT vs NOCOUPL spread onset timing by lead day",
    "Days after initialization",
    calendar_axis=False,
)
_plot_spread_onset(
    spread_onset_df,
    "onset_calendar_doy",
    "08_INT_vs_NOCOUPL_spread_onset_calendar_month_day_v1",
    "INT vs NOCOUPL spread onset timing by calendar date",
    "Calendar date",
    calendar_axis=True,
)


### 最终 synthesis matrix

### 目的
整合 01/03/04/05/07/08 的诊断成一个数值矩阵。

### 科学问题
机制链条哪些环节跨 case 稳健，哪些只适用于 0008-01 或 paired feedback？

### 方法
矩阵只使用数值和 NA，不使用 √、? 或 /。列包括 event severity、EP100-O3、W1+W2 dominance、spread lead、Z300 source、FWD delay、skill delta。

### 输出
final_mechanism_robustness_summary_matrix_v2.png/pdf 和 CSV。

### 解读
数值越强越支持相应机制；NA 表示不适用或没有数据。

### 注意
这是索引图，不应替代各 notebook 的具体图和表。


In [ ]:
fig_dir = figure_dir("99_synthesis")
tab_dir = table_dir("99_synthesis")
inv = discover_hindcast_cases()
matrix = inv[["case", "year", "init_month", "config", "perturbation"]].copy()
for col in ["MA_min_O3_DU", "EP100_O3_link_R", "W1W2_dominance_R", "spread_lead_O3_minus_dyn_days", "Z300_projection_EP100anom_R", "Z300_wave2_EP100anom_R", "INT_minus_NOCOUPL_FWD_days", "INT_minus_NOCOUPL_O3_RMSE_MA"]:
    matrix[col] = np.nan
sev_path = TAB_DIR / "01_reference_O3_severity_selected_years.csv"
if sev_path.exists():
    sev = pd.read_csv(sev_path, dtype={"year": str})
    sev["year"] = sev["year"].astype(str).str.zfill(4)
    sev_map = sev.set_index("year")["MA_min_O3_DU"].to_dict()
    matrix["MA_min_O3_DU"] = matrix["year"].astype(str).str.zfill(4).map(sev_map)
cor03 = TAB_DIR / "03_EP100_O3_U_FWD_correlations_all_cases.csv"
if cor03.exists():
    c03 = pd.read_csv(cor03, dtype={"case": str})
    for _, r in c03.iterrows():
        metric = r.get("metric", r.get("relationship", ""))
        if metric == "wave1_plus_wave2 vs O3_RMSE":
            matrix.loc[matrix["case"] == r["case"], "EP100_O3_link_R"] = r["R"]
        if metric == "all_waves vs W1+W2":
            matrix.loc[matrix["case"] == r["case"], "W1W2_dominance_R"] = r["R"]
spread_path = TAB_DIR / "04_spread_onset_timing_summary_all_cases.csv"
if spread_path.exists():
    sp = pd.read_csv(spread_path, dtype={"case": str})
    piv = sp.pivot_table(index="case", columns="variable", values="onset_lead_day", aggfunc="first")
    for case in piv.index:
        dyn = np.nanmin([piv.loc[case].get("EP100_W1W2", np.nan), piv.loc[case].get("U60N10", np.nan), piv.loc[case].get("U60N50", np.nan)])
        o3 = piv.loc[case].get("O3", np.nan)
        matrix.loc[matrix["case"] == case, "spread_lead_O3_minus_dyn_days"] = o3 - dyn if np.isfinite(o3) and np.isfinite(dyn) else np.nan
zcor_path = TAB_DIR / "05_Z300_source_metric_correlations_all_cases.csv"
if zcor_path.exists():
    zc = pd.read_csv(zcor_path, dtype={"case": str})
    # Use the average Jan/Feb relationship by case for synthesis.
    for case, sub in zc.groupby("case"):
        a = sub[sub["relationship"] == "Z300_stationary_projection_to_B2000 vs EP100_wave1_plus_wave2_anom_ref"]["R"]
        b = sub[sub["relationship"] == "Z300_wave2_amplitude_m vs EP100_wave2_anom_ref"]["R"]
        if len(a): matrix.loc[matrix["case"] == case, "Z300_projection_EP100anom_R"] = float(a.mean())
        if len(b): matrix.loc[matrix["case"] == case, "Z300_wave2_EP100anom_R"] = float(b.mean())
fwd_path = TAB_DIR / "08_FWD_INT_NOCOUPL_summary.csv"
if fwd_path.exists():
    fwd = pd.read_csv(fwd_path, dtype={"member": str, "year": str, "init_month": str})
    if "plev_hpa" in fwd.columns:
        fwd = fwd[np.isclose(fwd["plev_hpa"].astype(float), 50.0)]
    for (year, init), sub in fwd.groupby(["year", "init_month"]):
        means = sub.groupby("config")["FWD_DOY"].mean()
        if set(["INT", "NOCOUPL"]).issubset(means.index):
            delta = float(means["INT"] - means["NOCOUPL"])
            mask = (matrix["year"].astype(str).str.zfill(4) == str(year).zfill(4)) & (matrix["init_month"].astype(str).str.zfill(2) == str(init).zfill(2))
            matrix.loc[mask, "INT_minus_NOCOUPL_FWD_days"] = delta
skill_path = TAB_DIR / "08_INT_vs_NOCOUPL_skill_difference_heatmap.csv"
if skill_path.exists():
    sk = pd.read_csv(skill_path, dtype={"year": str, "init_month": str})
    sub = sk[sk["metric"] == "O3_RMSE_MarApr"]
    for _, r in sub.iterrows():
        mask = (matrix["year"].astype(str).str.zfill(4) == str(r["year"]).zfill(4)) & (matrix["init_month"].astype(str).str.zfill(2) == str(r["init_month"]).zfill(2))
        matrix.loc[mask, "INT_minus_NOCOUPL_O3_RMSE_MA"] = r["mean_INT_minus_NOCOUPL"]
out_csv = tab_dir / "final_mechanism_robustness_summary_matrix_v2.csv"
matrix.to_csv(out_csv, index=False)
matrix.to_csv(TAB_DIR / "final_mechanism_robustness_summary_matrix.csv", index=False)
plot_cols = ["MA_min_O3_DU", "EP100_O3_link_R", "W1W2_dominance_R", "spread_lead_O3_minus_dyn_days", "Z300_projection_EP100anom_R", "Z300_wave2_EP100anom_R", "INT_minus_NOCOUPL_FWD_days", "INT_minus_NOCOUPL_O3_RMSE_MA"]
vals = matrix[plot_cols].astype(float)
scaled = vals.copy()
for col in plot_cols:
    v = vals[col].values.astype(float)
    finite = np.isfinite(v)
    if finite.any():
        lo, hi = np.nanpercentile(v[finite], [5, 95])
        if hi > lo:
            scaled[col] = np.clip((v - lo) / (hi - lo) * 2 - 1, -1, 1)
        else:
            scaled[col] = 0
fig, ax = plt.subplots(figsize=(14, max(5, 0.34 * len(matrix) + 2)), constrained_layout=True)
im = ax.imshow(scaled[plot_cols].values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(plot_cols)), plot_cols, rotation=45, ha="right")
ax.set_yticks(range(len(matrix)), matrix["case"])
for i in range(len(matrix)):
    for j, col in enumerate(plot_cols):
        val = vals.iloc[i, j]
        ax.text(j, i, "NA" if not np.isfinite(val) else f"{val:.2g}", ha="center", va="center", fontsize=7)
ax.set_title("Mechanism robustness summary matrix (numeric values; NA = not applicable/missing)\nNo check marks: read each column by its physical sign and units")
fig.colorbar(im, ax=ax, label="column-wise scaled value for display only")
savefig(fig, "final_mechanism_robustness_summary_matrix_v2", fig_dir=fig_dir, notebook="08_final_warming_and_skill.ipynb", scientific_question="哪些机制证据跨 hindcast cases 稳健？", variables_windows="Compiled O3 severity, EP100, spread timing, Z300 Jan/Feb source, INT/NOCOUPL FWD and O3 skill", interpretation="图中文字是实际数值；颜色只是每列缩放后的显示辅助。NA 表示不适用或缺数据。", caveat="这是索引矩阵，必须回看各分图解释符号和窗口。", csv_table=out_csv)
plt.show()
write_figure_guide()
with (OUT_ROOT / "EXTENTION_ANALYSIS_SYNTHESIS.md").open("w") as f:
    f.write("""# Extention Analysis Synthesis\n\n0008-01 provides the detailed source mechanism:\nZ300 stationary-wave geometry / wave-2 amplitude -> constructive/destructive interference -> EP100 W1+W2 anomaly -> U60N10/U60N50 vortex pathway spread -> later O3 RMSE.\n\nThis revised workflow separates three interpretations:\n\n1. Jan/Feb source diagnosis: use Z300 stationary-wave metrics and EP100 reference anomalies only where the case actually contains Jan/Feb data.\n2. Post-initialization wave forcing: use primary/lead windows for all cases, especially later initialized cases.\n3. INT-vs-NOCOUPL feedback: use paired small-perturbation cases to test ozone feedback on O3, vortex wind, EP100, FWD, and skill.\n\nLater-initialized cases should not be interpreted as long-lead precursor tests. They are better suited for diagnosing early post-initialization wave forcing, vortex persistence, final warming, and feedback differences.\n""")
